In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
BASE_PATH = "/Volumes/main/lakehouse_marketing"
RAW_PATH = f"{BASE_PATH}/raw"
BRONZE_PATH = f"{BASE_PATH}/bronze"

%md
* Definindo Schema e lendo o csv da Raw

In [0]:
events_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("campaign_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("event_timestamp", StringType(), True)
])


df_events_raw = spark.read\
                    .schema(events_schema)\
                    .option("header", "true")\
                    .csv(f"{RAW_PATH}/events")

In [0]:
df_events_bronze = df_events_raw\
                        .withColumn("ingestion_timestamp", F.current_timestamp())\
                        .withColumn("source_file", F.col("_metadata.file_path"))


In [0]:
display(df_events_bronze.limit(5))

event_id,user_id,campaign_id,event_type,event_timestamp,ingestion_timestamp,source_file
deb8effe-a1ac-4fa8-b5f2-f2f1208bd214,243b8c14-31df-4eca-8bcf-0fcd72fc6c14,1987-07-20T15:38:13.627043,click,3d74bc62-c272-4fb3-bdaf-5fbe6148f26d,2026-01-19T13:37:58.625Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00003-tid-6666582444410329427-f3c55bad-6077-43b2-b9ab-03fee5f242b2-220-1-c000.csv
6cd37cfb-43a0-4539-8147-399f7c277cf8,c609fc4a-6aaf-45a4-8e22-a91f25a05389,1978-06-08T01:10:32.653571,click,d08257b8-51ca-4788-b6fb-2b8df242ba5f,2026-01-19T13:37:58.625Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00003-tid-6666582444410329427-f3c55bad-6077-43b2-b9ab-03fee5f242b2-220-1-c000.csv
524310a9-5487-4d84-979e-d3581c4cbc97,ab29f860-0552-4e32-b985-102948e77bf7,2025-07-08T08:39:11.871619,click,2589f83e-75ed-42f2-bb98-381c38be4ae9,2026-01-19T13:37:58.625Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00003-tid-6666582444410329427-f3c55bad-6077-43b2-b9ab-03fee5f242b2-220-1-c000.csv
baf821ba-be4e-41aa-b712-2eedc0848e85,96516a56-c5eb-49dc-8643-240f19ade065,1971-03-18T08:38:14.490600,click,4ea3a633-9a12-4457-bbbe-5485699084fb,2026-01-19T13:37:58.625Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00003-tid-6666582444410329427-f3c55bad-6077-43b2-b9ab-03fee5f242b2-220-1-c000.csv
89cc39f1-6180-4b0b-9ecc-28bd42805f34,fce3f071-0898-4529-9b07-8f94ca3fce62,1984-08-15T15:00:18.019250,click,1610cbeb-2ed7-49bb-817a-fd6e57e8be32,2026-01-19T13:37:58.625Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00003-tid-6666582444410329427-f3c55bad-6077-43b2-b9ab-03fee5f242b2-220-1-c000.csv


In [0]:
BRONZE_EVENTS_PATH = f"{BRONZE_PATH}/events"

df_events_bronze.write\
    .format('delta')\
    .mode('overwrite')\
    .save(BRONZE_EVENTS_PATH)

In [0]:
dbutils.fs.ls(BRONZE_EVENTS_PATH)

[FileInfo(path='dbfs:/Volumes/main/lakehouse_marketing/bronze/events/_delta_log/', name='_delta_log/', size=0, modificationTime=1768829910141),
 FileInfo(path='dbfs:/Volumes/main/lakehouse_marketing/bronze/events/part-00000-75fca8d3-320b-422c-a457-60171cf717d2.c000.snappy.parquet', name='part-00000-75fca8d3-320b-422c-a457-60171cf717d2.c000.snappy.parquet', size=12237760, modificationTime=1768829887000)]

In [0]:
display(spark.read\
    .format("delta")\
    .load(f"{BRONZE_EVENTS_PATH}"))

event_id,user_id,campaign_id,event_type,event_timestamp,ingestion_timestamp,source_file
bd9c66b3-ad3c-4d6d-9a3d-1fa7bc8960a9,bdd640fb-0667-4ad1-9c80-317fa3b1799d,2019-12-17T16:17:54.240000,view,23b8c1e9-3924-46de-beb1-3b9046685257,2026-01-19T13:38:04.980Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-6666582444410329427-f3c55bad-6077-43b2-b9ab-03fee5f242b2-214-1-c000.csv
8fadc1a6-06cb-4fb3-9a1d-e644815ef6d1,0822e8f3-6c03-4199-972a-846916419f82,1981-02-18T19:27:46.798518,view,3b8faa18-37f8-488b-97fc-695a07a0ca6e,2026-01-19T13:38:04.980Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-6666582444410329427-f3c55bad-6077-43b2-b9ab-03fee5f242b2-214-1-c000.csv
c241330b-01a9-471f-9e8a-774bcf36d58b,6b65a6a4-8b81-48f6-b38a-088ca65ed389,2015-02-15T08:35:56.806669,view,47378190-96da-4dac-b2ff-5d2a386ecbe0,2026-01-19T13:38:04.980Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-6666582444410329427-f3c55bad-6077-43b2-b9ab-03fee5f242b2-214-1-c000.csv
6142ea7d-17be-4111-9a2a-73ed562b0f79,47229389-571a-4876-ac30-7511b2b9437a,1975-06-02T03:10:48.916006,view,c37459ee-f50b-4a63-b71e-cd7b27cd8130,2026-01-19T13:38:04.980Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-6666582444410329427-f3c55bad-6077-43b2-b9ab-03fee5f242b2-214-1-c000.csv
759cde66-bacf-43d0-8b1f-9163ce9ff57f,43b7a3a6-9a8d-4a03-980d-7b71d8f56413,2000-01-11T10:21:06.373450,view,null,2026-01-19T13:38:04.980Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-6666582444410329427-f3c55bad-6077-43b2-b9ab-03fee5f242b2-214-1-c000.csv
5c941cf0-dc98-42c1-a2ac-f72f9e574f7a,142c3fe8-60e7-4113-ac1b-8ca1f91e1d4c,2002-05-01T12:55:16.733859,view,a0ee89ae-d453-4d32-8b0d-bb418d5288f1,2026-01-19T13:38:04.980Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-6666582444410329427-f3c55bad-6077-43b2-b9ab-03fee5f242b2-214-1-c000.csv
ddd1dfb2-3b98-4ef8-9af6-1a26146d3f31,a9488d99-0bbb-4599-91ce-5dd2b45ed1f0,1975-08-28T18:35:04.416331,click,fc377a4c-4a15-444d-85e7-ce8a3a578a8e,2026-01-19T13:38:04.980Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-6666582444410329427-f3c55bad-6077-43b2-b9ab-03fee5f242b2-214-1-c000.csv
b3aa7efe-4458-4885-ab90-99a435a240ae,d58842de-a2bc-472f-b412-b29347294739,2022-06-14T07:45:25.754665,click,5af30553-5ec4-4e08-a9a3-b2e95d65a441,2026-01-19T13:38:04.980Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-6666582444410329427-f3c55bad-6077-43b2-b9ab-03fee5f242b2-214-1-c000.csv
451b4cf3-6123-4df7-b656-af7229d4beef,a28defe3-9bf0-4273-9247-6f57a5e5a5ab,2025-05-30T15:27:12.837889,view,3eabedcb-baa8-4dd4-88bd-64072bcfbe01,2026-01-19T13:38:04.980Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-6666582444410329427-f3c55bad-6077-43b2-b9ab-03fee5f242b2-214-1-c000.csv
d261a7ab-3aa2-44f9-8e51-f30dc6a7ee39,3838b326-8e94-4239-b02b-61c4a3d70628,1971-10-19T13:03:02.976742,view,c4b032cc-d7c5-44a5-9304-317faf42e12f,2026-01-19T13:38:04.980Z,dbfs:/Volumes/main/lakehouse_marketing/raw/events/part-00000-tid-6666582444410329427-f3c55bad-6077-43b2-b9ab-03fee5f242b2-214-1-c000.csv


In [0]:
spark.read\
    .format("delta")\
    .load(f"{BRONZE_EVENTS_PATH}")\
    .count()

100000